# Título épico (nose q poner todavía)
> **Integrantes:** Matías Soto, Matías Toledo

Esta tarea trata sobre blablabla

### Librerías importantes para esta tarea

In [3]:
import torch.nn as nn
import torch
import pandas as pd

In [4]:
data = pd.read_csv("Pokemon.csv")
data.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


### Explicación del dataset

El dataset contiene variables numéricas de estadísticas de los pokemons, como `HP` (Hit Points o puntos de salud), `Attack` (Ataque, daño de los movimientos físicos), `Defense` (Defensa, resistencia ante movimientos físicos), `Sp. Atk` (Ataque Especial, potencia de movimientos especiales), `Sp. Def`(Defensa Especial, resistencia al recibir movimientos de Ataque Especial) y `Speed` (Velocidad, determina si ataca antes o después que el oponente).

Además de variables categóricas como `Type 1` y `Type 2` (Tipo, clasificación elemental que define sus habilidades, debilidades y resistencias), `Generation` (Generación de pokemon a la que pertenece), y finalmente la variable objetivo `Legendary` (Si el pokemon es legendario o no).

Para efectos de este trabajo, se tiene el siguiente análisis de las variables del dataset y su aporte al aprendizaje del modelo:

- La columna `#` solo enumera los pokemons y no aporta información útil.

- La columna `Name` es un identificador ùnico de cada pokemon y no aporta información útil.

- Las variables numéricas (`HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def` y `Speed`) si aportan información y no requieren cambios.

- `Type 1` y `Type 2` sí aportan información pero requieren codificación numérica. 

- `Generation` se mantendrá para evaluar si agrega señal al modelo (puede llegar a causar sesgo por número de legendarios pertenecientes a cada generación). 

- La variable objetivo `Legendary` se transformará a binaria (0/1).

### Preparación del dataset

1. Se elimina `Name` por ser un identificador sin valor predictivo.
2. Se convierten `Type 1` y `Type 2` a códigos numéricos (en `Type 2`, `NaN` queda como el código 18).
3. Se transforma `Legendary` a 0/1 para un problema de clasificación binaria.
4. Se mantiene `Generation` para evaluar su desempeño en el modelo.

Luego se normaliza para que todas las variables estén entre 0 y 1

In [5]:
df = data.copy()

# Variable objetivo binaria
df["Legendary"] = df["Legendary"].astype(int)

# Definir categorias para Type 1 y Type 2
type1_cat = pd.Categorical(df["Type 1"])
type1_categories = list(type1_cat.categories)
type_categories = type1_categories + ["None"]

# Codificacion de Type 1 y mapeo codigo->categoria
print("Type 1 mapeo codigo->categoria:", dict(enumerate(type_categories)))
df["Type 1"] = pd.Categorical(df["Type 1"], categories=type_categories).codes

# Relleno de los faltantes en Type 2 (NaN) por None
type2 = df["Type 2"].fillna("None")

# Codificacion de Type 2 y mapeo codigo->categoria
print("Type 2 mapeo codigo->categoria:", dict(enumerate(type_categories)))
df["Type 2"] = pd.Categorical(type2, categories=type_categories).codes


Type 1 mapeo codigo->categoria: {0: 'Bug', 1: 'Dark', 2: 'Dragon', 3: 'Electric', 4: 'Fairy', 5: 'Fighting', 6: 'Fire', 7: 'Flying', 8: 'Ghost', 9: 'Grass', 10: 'Ground', 11: 'Ice', 12: 'Normal', 13: 'Poison', 14: 'Psychic', 15: 'Rock', 16: 'Steel', 17: 'Water', 18: 'None'}
Type 2 mapeo codigo->categoria: {0: 'Bug', 1: 'Dark', 2: 'Dragon', 3: 'Electric', 4: 'Fairy', 5: 'Fighting', 6: 'Fire', 7: 'Flying', 8: 'Ghost', 9: 'Grass', 10: 'Ground', 11: 'Ice', 12: 'Normal', 13: 'Poison', 14: 'Psychic', 15: 'Rock', 16: 'Steel', 17: 'Water', 18: 'None'}


In [13]:
# Separación de variables y objetivo
X = df.drop(columns=["#","Name", "Legendary"])
y = df["Legendary"]

X_tensor = torch.tensor(X.values, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.long)  # para CrossEntropyLoss

X.head()

,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
0,9,13,318,45,49,49,65,65,45,1
1,9,13,405,60,62,63,80,80,60,1
2,9,13,525,80,82,83,100,100,80,1
3,9,13,625,80,100,123,122,120,80,1
4,6,18,309,39,52,43,60,50,65,1


In [7]:
# Normalización df

### Definición del modelo
Definición épica

Función de activación sigmoide

input_dim = 10, correspondiente a las 10 variables a trabajar
hidden_dim = 8, ajustable, 8 para empezar
output_dim = 2, clasificamos 2 categorias, "legendario" y "no legendario"

In [8]:
class MultiLayerPerceptron(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim): 
        super(type(self), self).__init__()  
        # Capa oculta
        self.hidden = nn.Linear(input_dim, hidden_dim)
        # Capa de salida
        self.output = nn.Linear(hidden_dim, output_dim) 
        # Función de activación sigmoide   
        self.activation = nn.Sigmoid()
        
    def forward(self, x):
        # Conexión 
        x = self.activation(self.hidden(x))
        return self.output(x)
    
# input_dim = 10, correspondiente a las 10 variables a trabajar
# hidden_dim = 8, ajustable, 8 para empezar
# output_dim = 2, clasificamos 2 categorias, "legendario" y "no legendario"
modelo = MultiLayerPerceptron(input_dim= 10, hidden_dim= 8, output_dim= 2)

### Definición de optimizador y función de costo

Textos copiados notebook ex-profe hjuise, revisar y cambiar

#### Optimizador

`Adam`: Gradiente descedente con tasa de aprendizaje adaptiva

- `lr` Es la tasa de aprendizaje. Debe ser un valor pequeño para no desestabilizar el entrenamiento, pero no tan pequeño para enlentencerlo demasiado.

- `momentum` es la tasa de momentum. Podemos utilizar un valor mayor que cero para evitar estancamiento en mínimos locales.

- `weight_decay` controla la regularización (norma L2) de los parámetros. Podemos utilizar un valor mayor que cero para evitar sobreajuste.

#### Función de Costo
`CrossEntropyLoss` sirve por ser 
$$
\mathcal{L}(y, \hat y) = \sum_{d=1}^D ( y_d - \hat y_d)^2
$$

donde $$y_c \in \{0,1\}, \hat y_c \in [0,1] y \sum_{c=1}^C \hat y_c = 1$$

Esta función de costo se utiliza en problemas de clasificación de C clases. Cuando entrenamos con esta función de costo debemos asegurarnos de que el modelo tenga tantas unidades de salida como clases tenga el problema.

In [20]:
# Optimizador
optimizer = torch.optim.SGD(modelo.parameters(), lr=1e-3, momentum=0, weight_decay=0)

# Funcion de costo
criterion = torch.nn.CrossEntropyLoss(reduction='mean')

In [19]:
loss = criterion(X_tensor, y_tensor)
loss

tensor(425.4475)

### Entrenamiento del modelo
Entrenamiento épico

In [17]:
modelo

MultiLayerPerceptron(
  (hidden): Linear(in_features=10, out_features=8, bias=True)
  (output): Linear(in_features=8, out_features=2, bias=True)
  (activation): Sigmoid()
)

In [16]:
loss = criterion(X_tensor, y_tensor)
loss

tensor(425.4475)

### Evaluación del modelo
Evaluación épica

In [ ]:
modelo.evaluar(1.0)

### Preguntas finales
1. Sobre la matriz de confusión, interprete los resultados obtenidos. Con sus palabras defina que significa cada tipo de error. ¿Elegiría a Pokémon ubicados en FP o FN para su equipo?
2. Busque un caso mal clasificado por el modelo, e interprete por qué cree que el modelo se equivocó en ese caso.
3. ¿Cúal fue el mayor desafío que enfrentó al realizar esta tarea? ¿Cómo lo solucionó?


### IA Generativa
1. ¿Utilizó alguna herramienta de IA Generativa para realizar esta tarea? En caso afirmativo, indique cuál o cuáles herramientas utilizó.
2. ¿En qué parte o partes de la tarea utilizó estas herramientas?